<a href="https://colab.research.google.com/github/aatayo/aatayo.github.io/blob/main/P1_Physics_Informed_Neural_Network_(PINN)_for_Cooling_Fin_Heat_Transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# Physics-Informed Neural Network (PINN) for Cooling Fin Heat Transfer
# =============================================================================
# This script implements a PINN to solve the transient heat conduction PDE
# for a 1D aluminum cooling fin with convective heat loss along its length.
# The network learns to satisfy the governing physics, initial conditions,
# and boundary conditions simultaneously through a composite loss function.
# =============================================================================

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import seaborn as sns
from scipy.integrate import odeint
import time
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for clean output

# Set plotting style and resolution
plt.style.use('default')
plt.rcParams['figure.dpi'] = 100       # Screen display resolution
plt.rcParams['savefig.dpi'] = 300      # High resolution for saved figures

# Set random seeds to ensure reproducibility across runs
np.random.seed(42)
tf.random.set_seed(42)

# =============================================================================
# PHYSICAL PARAMETERS (Aluminum Fin)
# =============================================================================
rho   = 2700.0   # Density [kg/m³] - material property of aluminum
c_p   = 900.0    # Specific heat capacity [J/(kg·K)] - energy storage per unit mass
k     = 200.0    # Thermal conductivity [W/(m·K)] - heat conduction ability
h     = 50.0     # Convective heat transfer coefficient [W/(m²·K)] - surface cooling rate
P     = 0.1      # Fin perimeter [m] - cross-sectional perimeter for convective area
T_inf = 298.0    # Ambient (surrounding) temperature [K] = 25°C
L     = 1.0      # Fin length [m] - spatial domain extent
t_max = 1000.0   # Total simulation time [s]

# =============================================================================
# INITIAL AND BOUNDARY TEMPERATURES
# =============================================================================
T_initial = 373.0  # Initial fin temperature [K] = 100°C (uniform throughout fin)
T_base    = 373.0  # Fixed base temperature [K] = 100°C (Dirichlet BC at x=0)

# =============================================================================
# DERIVED THERMAL PARAMETERS
# =============================================================================
# Thermal diffusivity: ratio of conductive to storage capacity
alpha = k / (rho * c_p)       # [m²/s] - governs how fast heat diffuses

# Convective decay parameter: rate of heat loss to surroundings
beta  = h * P / (rho * c_p)   # [1/s] - governs convective cooling rate

print(f"Thermal diffusivity α = {alpha:.2e} m²/s")
print(f"Convective parameter β = {beta:.2e} 1/s")

# =============================================================================
# PINN CLASS DEFINITION
# =============================================================================
class CoolingFinPINN:
    def __init__(self, layers, lb, ub):
        """
        Initialize the Physics-Informed Neural Network.

        Args:
            layers : List[int] - defines network architecture,
                     e.g. [2, 64, 64, 64, 64, 1] means 2 inputs,
                     4 hidden layers of 64 neurons, and 1 output (temperature)
            lb     : List[float] - lower bounds of input domain [x_min, t_min]
            ub     : List[float] - upper bounds of input domain [x_max, t_max]
        """
        self.layers = layers

        # Store domain bounds as TensorFlow constants for use in normalization
        self.lb = tf.constant(lb, dtype=tf.float32)
        self.ub = tf.constant(ub, dtype=tf.float32)

        # Temperature scaling constants to help network learn in a normalized range
        self.T_scale = (T_initial + T_inf) / 2.0   # Midpoint temperature
        self.T_range = T_initial - T_inf            # Temperature span

        # Initialize all network weights and biases
        self.weights, self.biases = self.initialize_network()

        # Containers to track different loss components during training
        self.loss_history     = []   # Total weighted loss
        self.pde_loss_history = []   # Physics (PDE) residual loss
        self.ic_loss_history  = []   # Initial condition loss
        self.bc_loss_history  = []   # Boundary condition loss

    def initialize_network(self):
        """
        Initialize network weights and biases using Xavier (Glorot) initialization.
        Xavier initialization sets weights proportional to the harmonic mean of
        the number of input and output neurons, preventing vanishing/exploding gradients.

        Returns:
            weights : List of TensorFlow Variable weight matrices
            biases  : List of TensorFlow Variable bias vectors
        """
        weights = []
        biases  = []

        for i in range(len(self.layers) - 1):
            # Xavier initialization: scale factor based on layer sizes
            w = tf.Variable(
                tf.random.normal(
                    [self.layers[i], self.layers[i+1]], dtype=tf.float32
                ) * np.sqrt(2.0 / (self.layers[i] + self.layers[i+1])),
                trainable=True
            )
            # Biases initialized to zero
            b = tf.Variable(
                tf.zeros([self.layers[i+1]], dtype=tf.float32),
                trainable=True
            )
            weights.append(w)
            biases.append(b)

        return weights, biases

    def neural_net(self, X):
        """
        Perform a forward pass through the neural network.

        Inputs are normalized to [-1, 1] to improve training stability.
        Hidden layers use tanh activation for smooth, differentiable outputs
        (required for computing PDE derivatives via automatic differentiation).
        Output is scaled to the physical temperature range using sigmoid activation.

        Args:
            X : tf.Tensor of shape [N, 2] - concatenated [x, t] inputs

        Returns:
            Y : tf.Tensor of shape [N, 1] - predicted temperature T(x, t)
        """
        # Normalize inputs from physical domain to [-1, 1]
        X_norm = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0

        H = X_norm
        # Apply tanh activation through all hidden layers
        for i in range(len(self.weights) - 1):
            H = tf.tanh(tf.add(tf.matmul(H, self.weights[i]), self.biases[i]))

        # Raw output from the final linear layer
        Y_raw = tf.add(tf.matmul(H, self.weights[-1]), self.biases[-1])

        # Scale output to physically meaningful temperature range [T_inf, T_initial]
        # Sigmoid ensures output stays within bounds: T_inf ≤ T ≤ T_initial
        Y = T_inf + (T_initial - T_inf) * tf.sigmoid(Y_raw)

        return Y

    def physics_net(self, x, t):
        """
        Compute the PDE residual using automatic differentiation.

        The governing PDE for the fin is:
            ρ·cₚ · ∂T/∂t = k · ∂²T/∂x² - h·P · (T - T∞)

        This residual should be zero everywhere in the domain if the network
        has correctly learned the physics. The nested GradientTapes allow
        computation of both first and second-order derivatives.

        Args:
            x : tf.Tensor [N, 1] - spatial positions
            t : tf.Tensor [N, 1] - time values

        Returns:
            pde_residual : tf.Tensor [N, 1] - PDE violation at each point
            T            : tf.Tensor [N, 1] - predicted temperatures
        """
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch([x, t])
            with tf.GradientTape(persistent=True) as tape1:
                tape1.watch([x, t])
                X = tf.concat([x, t], axis=1)
                T = self.neural_net(X)           # Predicted temperature

            T_x = tape1.gradient(T, x)           # First spatial derivative ∂T/∂x
            T_t = tape1.gradient(T, t)           # First time derivative ∂T/∂t

        T_xx = tape2.gradient(T_x, x)            # Second spatial derivative ∂²T/∂x²

        # Release persistent tapes to free memory
        del tape1, tape2

        # PDE residual: should be zero if physics are satisfied
        pde_residual = rho * c_p * T_t - k * T_xx + h * P * (T - T_inf)

        return pde_residual, T

    def get_trainable_variables(self):
        """
        Return all trainable parameters (weights + biases) as a flat list.
        Used by the optimizer to compute and apply gradients.
        """
        return self.weights + self.biases

    @tf.function  # Compile as TensorFlow graph for faster execution
    def loss_function(self, x_pde, t_pde, x_ic, t_ic, T_ic,
                      x_bc1, t_bc1, T_bc1, x_bc2, t_bc2):
        """
        Compute the weighted composite loss function combining:
            1. PDE loss     - physics residual across the domain interior
            2. IC loss      - error at t=0 (initial temperature condition)
            3. BC loss      - error at x=0 (fixed base) and x=L (convective tip)

        Loss weights are tuned to balance contributions from each term,
        accounting for the large magnitude differences in thermal values.

        Args:
            x_pde, t_pde   : Collocation points in the domain interior
            x_ic, t_ic     : Points at t=0 for initial condition enforcement
            T_ic           : Known initial temperatures at IC points
            x_bc1, t_bc1   : Points at x=0 (base boundary)
            T_bc1          : Known base temperatures (fixed at T_base)
            x_bc2, t_bc2   : Points at x=L (tip boundary)

        Returns:
            loss_total : Scalar weighted total loss
            loss_pde   : Unweighted PDE residual loss
            loss_ic    : Unweighted initial condition loss
            loss_bc    : Unweighted combined boundary condition loss
        """
        # --- PDE Loss: enforce heat equation at interior collocation points ---
        pde_residual, _ = self.physics_net(x_pde, t_pde)
        loss_pde = tf.reduce_mean(tf.square(pde_residual))

        # --- Initial Condition Loss: T(x, 0) = T_initial ---
        X_ic       = tf.concat([x_ic, t_ic], axis=1)
        T_pred_ic  = self.neural_net(X_ic)
        loss_ic    = tf.reduce_mean(tf.square(T_pred_ic - T_ic))

        # --- Boundary Condition 1: T(0, t) = T_base (Dirichlet BC at base) ---
        X_bc1      = tf.concat([x_bc1, t_bc1], axis=1)
        T_pred_bc1 = self.neural_net(X_bc1)
        loss_bc1   = tf.reduce_mean(tf.square(T_pred_bc1 - T_bc1))

        # --- Boundary Condition 2: Convective tip at x=L ---
        # Robin BC: k · ∂T/∂x|_{x=L} = -h · (T(L,t) - T∞)
        # Heat conducted to tip equals heat convected away at tip surface
        with tf.GradientTape() as tape:
            tape.watch(x_bc2)
            X_bc2      = tf.concat([x_bc2, t_bc2], axis=1)
            T_pred_bc2 = self.neural_net(X_bc2)
        T_x_bc2 = tape.gradient(T_pred_bc2, x_bc2)   # ∂T/∂x at x=L

        bc2_residual = k * T_x_bc2 + h * (T_pred_bc2 - T_inf)
        loss_bc2     = tf.reduce_mean(tf.square(bc2_residual))

        # --- Loss Weights ---
        # PDE loss scaled down due to large thermal magnitude values
        # IC and BC losses kept at unit scale for direct enforcement
        w_pde = 1e-8   # Downscale PDE contribution to balance magnitudes
        w_ic  = 1.0    # Full weight on initial condition
        w_bc  = 1.0    # Full weight on boundary conditions

        # Weighted sum of all loss components
        loss_total = w_pde * loss_pde + w_ic * loss_ic + w_bc * (loss_bc1 + loss_bc2)

        return loss_total, loss_pde, loss_ic, (loss_bc1 + loss_bc2)

    def train_step(self, optimizer, x_pde, t_pde, x_ic, t_ic, T_ic,
                   x_bc1, t_bc1, T_bc1, x_bc2, t_bc2):
        """
        Execute a single gradient descent update step.

        Computes the total loss, differentiates it with respect to all
        trainable parameters, and applies the optimizer update.

        Args:
            optimizer : tf.keras.optimizers instance (Adam)
            (remaining args): training data tensors passed to loss_function

        Returns:
            Scalar loss values for logging
        """
        with tf.GradientTape() as tape:
            loss_total, loss_pde, loss_ic, loss_bc = self.loss_function(
                x_pde, t_pde, x_ic, t_ic, T_ic,
                x_bc1, t_bc1, T_bc1, x_bc2, t_bc2
            )

        # Compute gradients of total loss w.r.t. all network parameters
        gradients = tape.gradient(loss_total, self.get_trainable_variables())

        # Apply gradients to update weights and biases
        optimizer.apply_gradients(zip(gradients, self.get_trainable_variables()))

        return loss_total, loss_pde, loss_ic, loss_bc

    def train(self, x_pde, t_pde, x_ic, t_ic, T_ic, x_bc1, t_bc1, T_bc1,
              x_bc2, t_bc2, epochs=20000, lr=0.001):
        """
        Full training loop with adaptive learning rate scheduling.

        Uses exponential decay to reduce learning rate over time, allowing
        large initial steps for rapid convergence followed by fine-tuning.

        Args:
            (data tensors): training points for PDE, IC, and BC enforcement
            epochs : int - total number of training iterations
            lr     : float - initial learning rate

        Returns:
            loss_history : List of total loss values per epoch
        """
        # Exponential decay: lr reduces by 5% every 1000 steps
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=lr,
            decay_steps=1000,
            decay_rate=0.95,
            staircase=True    # Step-wise decay (not continuous)
        )
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        print("Starting PINN training...")
        start_time = time.time()

        for epoch in range(epochs):
            # Perform one gradient update step
            loss_total, loss_pde, loss_ic, loss_bc = self.train_step(
                optimizer, x_pde, t_pde, x_ic, t_ic, T_ic,
                x_bc1, t_bc1, T_bc1, x_bc2, t_bc2
            )

            # Record loss components for convergence analysis
            self.loss_history.append(float(loss_total.numpy()))
            self.pde_loss_history.append(float(loss_pde.numpy()))
            self.ic_loss_history.append(float(loss_ic.numpy()))
            self.bc_loss_history.append(float(loss_bc.numpy()))

            # Print progress every 2000 epochs
            if epoch % 2000 == 0:
                elapsed = time.time() - start_time
                print(f"Epoch {epoch:5d}: Loss = {loss_total:.2e}, "
                      f"PDE = {loss_pde:.2e}, IC = {loss_ic:.2e}, "
                      f"BC = {loss_bc:.2e}, Time = {elapsed:.1f}s")

        training_time = time.time() - start_time
        print(f"\nTraining completed in {training_time:.1f} seconds")
        return self.loss_history

    def predict(self, x, t):
        """
        Generate temperature predictions for given spatial and temporal inputs.

        Args:
            x : tf.Tensor [N, 1] - spatial positions
            t : tf.Tensor [N, 1] - time values

        Returns:
            T : tf.Tensor [N, 1] - predicted temperatures T(x, t)
        """
        X = tf.concat([x, t], axis=1)
        return self.neural_net(X)

# =============================================================================
# TRAINING DATA GENERATION
# =============================================================================
def generate_training_data():
    """
    Generate randomized collocation and boundary/initial condition points.

    Rather than using a structured grid, points are sampled randomly across
    the domain. This is a key feature of PINNs — physics enforcement at
    arbitrary locations encourages global solution accuracy.

    Returns:
        data : dict of tf.Tensor objects for each training component
    """

    # --- PDE Collocation Points (interior of domain) ---
    # Randomly distributed across the full (x, t) domain
    n_pde = 10000
    x_pde = np.random.uniform(0, L,     (n_pde, 1))
    t_pde = np.random.uniform(0, t_max, (n_pde, 1))

    # --- Initial Condition Points: T(x, 0) = T_initial ---
    # All points at t=0, uniformly distributed along fin length
    n_ic  = 1000
    x_ic  = np.random.uniform(0, L, (n_ic, 1))
    t_ic  = np.zeros((n_ic, 1))                      # t = 0 for all IC points
    T_ic  = T_initial * np.ones((n_ic, 1))           # Uniform initial temperature

    # --- Boundary Condition 1: Fixed base temperature T(0, t) = T_base ---
    # Points located at x=0, distributed over all time
    n_bc1  = 500
    x_bc1  = np.zeros((n_bc1, 1))                    # x = 0 (base of fin)
    t_bc1  = np.random.uniform(0, t_max, (n_bc1, 1))
    T_bc1  = T_base * np.ones((n_bc1, 1))            # Fixed base temperature

    # --- Boundary Condition 2: Convective tip at x=L ---
    # Points located at x=L, distributed over all time
    # No explicit temperature specified — condition enforced via flux balance
    n_bc2  = 500
    x_bc2  = L * np.ones((n_bc2, 1))                 # x = L (tip of fin)
    t_bc2  = np.random.uniform(0, t_max, (n_bc2, 1))

    # Convert all numpy arrays to TensorFlow float32 tensors for GPU compatibility
    data = {
        'x_pde':  tf.constant(x_pde,  dtype=tf.float32),
        't_pde':  tf.constant(t_pde,  dtype=tf.float32),
        'x_ic':   tf.constant(x_ic,   dtype=tf.float32),
        't_ic':   tf.constant(t_ic,   dtype=tf.float32),
        'T_ic':   tf.constant(T_ic,   dtype=tf.float32),
        'x_bc1':  tf.constant(x_bc1,  dtype=tf.float32),
        't_bc1':  tf.constant(t_bc1,  dtype=tf.float32),
        'T_bc1':  tf.constant(T_bc1,  dtype=tf.float32),
        'x_bc2':  tf.constant(x_bc2,  dtype=tf.float32),
        't_bc2':  tf.constant(t_bc2,  dtype=tf.float32),
    }

    print(f"Generated training data:")
    print(f"  PDE points: {n_pde}")
    print(f"  IC points:  {n_ic}")
    print(f"  BC points:  {n_bc1 + n_bc2}")

    return data

# =============================================================================
# VISUALIZATION FUNCTIONS
# =============================================================================
def create_visualizations(model):
    """
    Generate five comprehensive plots to visualize PINN results:
        1. Temperature profiles along the fin at selected time snapshots
        2. 2D spatiotemporal heatmap with contour lines
        3. Training convergence (total, PDE, IC, BC losses)
        4. Temporal temperature evolution at fixed spatial positions
        5. 3D surface plot of the full temperature field T(x, t)

    Args:
        model : Trained CoolingFinPINN instance
    """

    # Define test grid for prediction
    x_test = np.linspace(0, L, 101)
    t_test = np.array([0, 50, 100, 200, 400, 600, 800, 1000])  # Selected time snapshots

    # Create meshgrid and flatten for batch prediction
    X_test, T_test = np.meshgrid(x_test, t_test)
    X_flat = X_test.flatten()[:, None]
    T_flat = T_test.flatten()[:, None]

    # Get PINN predictions and reshape back to grid
    T_pred = model.predict(
        tf.constant(X_flat, dtype=tf.float32),
        tf.constant(T_flat, dtype=tf.float32)
    ).numpy()
    T_pred = T_pred.reshape(X_test.shape)

    # -------------------------------------------------------------------------
    # Figure 1: Temperature Distribution at Selected Time Steps
    # -------------------------------------------------------------------------
    plt.figure(figsize=(14, 10))
    colors = plt.cm.coolwarm(np.linspace(0, 1, len(t_test)))  # Color gradient by time

    for i, t_val in enumerate(t_test):
        plt.plot(x_test, T_pred[i, :], 'o-', color=colors[i],
                 linewidth=3, markersize=6, label=f't = {t_val} s', alpha=0.8)

    plt.xlabel('Position x (m)', fontsize=16, fontweight='bold')
    plt.ylabel('Temperature T (K)', fontsize=16, fontweight='bold')
    plt.title('Temperature Distribution Along Cooling Fin\n(PINN Solution)',
              fontsize=18, fontweight='bold', pad=20)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.ylim([290, 380])

    # Add secondary Celsius axis on the right for engineering readability
    ax1         = plt.gca()
    ax1_celsius = ax1.twinx()
    ax1_celsius.set_ylabel('Temperature (°C)', fontsize=16, fontweight='bold')
    ax1_celsius.set_ylim([T - 273.15 for T in ax1.get_ylim()])

    plt.tight_layout()
    plt.savefig('cooling_fin_temperature_evolution.png', dpi=300, bbox_inches='tight')
    plt.show()


    # -------------------------------------------------------------------------
    # Figure 2: 2D Spatiotemporal Heatmap
    # -------------------------------------------------------------------------
    plt.figure(figsize=(14, 10))

    # Fine grid for smooth contour visualization
    x_fine = np.linspace(0, L, 200)
    t_fine = np.linspace(0, t_max, 150)
    X_fine, T_fine = np.meshgrid(x_fine, t_fine)

    X_fine_flat = X_fine.flatten()[:, None]
    T_fine_flat = T_fine.flatten()[:, None]

    T_pred_fine = model.predict(
        tf.constant(X_fine_flat, dtype=tf.float32),
        tf.constant(T_fine_flat, dtype=tf.float32)
    ).numpy()
    T_pred_fine = T_pred_fine.reshape(X_fine.shape)

    # Filled contour plot spanning full temperature range
    levels = np.linspace(T_inf, T_initial, 20)
    im     = plt.contourf(X_fine, T_fine, T_pred_fine, levels=levels,
                          cmap='coolwarm', extend='both')
    cbar   = plt.colorbar(im, shrink=0.8)
    cbar.set_label('Temperature (K)', fontsize=16, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)

    # Overlay contour lines with temperature labels
    contours = plt.contour(X_fine, T_fine, T_pred_fine, levels=10,
                           colors='black', alpha=0.4, linewidths=1)
    plt.clabel(contours, inline=True, fontsize=10, fmt='%.0f K')

    plt.xlabel('Position x (m)', fontsize=16, fontweight='bold')
    plt.ylabel('Time t (s)', fontsize=16, fontweight='bold')
    plt.title('Temperature Evolution Heatmap\n(PINN Solution)',
              fontsize=18, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig('cooling_fin_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

    # -------------------------------------------------------------------------
    # Figure 3: Training Loss Convergence (4 subplots)
    # -------------------------------------------------------------------------

    fig, ((ax31, ax32), (ax33, ax34)) = plt.subplots(2, 2, figsize=(16, 12))
    epochs = range(len(model.loss_history))

    # Total loss on log scale to visualize orders-of-magnitude reduction
    ax31.semilogy(epochs, model.loss_history, 'b-', linewidth=2)
    ax31.set_xlabel('Epoch', fontsize=14)
    ax31.set_ylabel('Total Loss', fontsize=14)
    ax31.set_title('Total Loss Convergence', fontsize=16, fontweight='bold')
    ax31.grid(True, alpha=0.3)

    # PDE (physics) residual loss
    ax32.semilogy(epochs, model.pde_loss_history, 'r-', linewidth=2)
    ax32.set_xlabel('Epoch', fontsize=14)
    ax32.set_ylabel('PDE Loss', fontsize=14)
    ax32.set_title('Physics Loss Convergence', fontsize=16, fontweight='bold')
    ax32.grid(True, alpha=0.3)

    # Initial condition loss
    ax33.semilogy(epochs, model.ic_loss_history, 'g-', linewidth=2)
    ax33.set_xlabel('Epoch', fontsize=14)
    ax33.set_ylabel('Initial Condition Loss', fontsize=14)
    ax33.set_title('IC Loss Convergence', fontsize=16, fontweight='bold')
    ax33.grid(True, alpha=0.3)

    # Boundary condition loss
    ax34.semilogy(epochs, model.bc_loss_history, 'm-', linewidth=2)
    ax34.set_xlabel('Epoch', fontsize=14)
    ax34.set_ylabel('Boundary Condition Loss', fontsize=14)
    ax34.set_title('BC Loss Convergence', fontsize=16, fontweight='bold')
    ax34.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('cooling_fin_training_convergence.png', dpi=300, bbox_inches='tight')
    plt.show()


    # -------------------------------------------------------------------------
    # Figure 4: Temperature vs Time at Fixed Spatial Positions
    # -------------------------------------------------------------------------
    plt.figure(figsize=(14, 10))

    positions   = [0.0, 0.25, 0.5, 0.75, 1.0]   # Positions along fin [m]
    t_continuous = np.linspace(0, t_max, 1000)    # Continuous time axis
    colors      = plt.cm.viridis(np.linspace(0, 1, len(positions)))

    for i, pos in enumerate(positions):
        # Predict temperature at fixed x = pos over all time
        x_pos = pos * np.ones((len(t_continuous), 1))
        t_pos = t_continuous.reshape(-1, 1)

        T_pred_pos = model.predict(
            tf.constant(x_pos, dtype=tf.float32),
            tf.constant(t_pos, dtype=tf.float32)
        ).numpy()

        plt.plot(t_continuous, T_pred_pos.flatten(), color=colors[i],
                 linewidth=3, label=f'x = {pos:.2f} m')

    plt.xlabel('Time t (s)', fontsize=16, fontweight='bold')
    plt.ylabel('Temperature T (K)', fontsize=16, fontweight='bold')
    plt.title('Temperature Evolution at Different Positions\n(PINN Solution)',
              fontsize=18, fontweight='bold', pad=20)

    # Dashed horizontal line indicating ambient temperature (equilibrium target)
    plt.axhline(y=T_inf, color='red', linestyle='--', linewidth=2,
                alpha=0.7, label='Ambient T∞')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('cooling_fin_temporal_evolution.png', dpi=300, bbox_inches='tight')
    plt.show()

    # -------------------------------------------------------------------------
    # Figure 5: 3D Surface Plot of Temperature Field T(x, t)
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=(14, 10))
    ax  = fig.add_subplot(111, projection='3d')

    # Coarser grid to keep 3D rendering manageable
    x_3d = np.linspace(0, L, 50)
    t_3d = np.linspace(0, t_max, 50)
    X_3d, T_3d = np.meshgrid(x_3d, t_3d)

    X_3d_flat = X_3d.flatten()[:, None]
    T_3d_flat = T_3d.flatten()[:, None]

    T_pred_3d = model.predict(
        tf.constant(X_3d_flat, dtype=tf.float32),
        tf.constant(T_3d_flat, dtype=tf.float32)
    ).numpy()
    T_pred_3d = T_pred_3d.reshape(X_3d.shape)

    # Plot 3D surface with coolwarm colormap
    surf = ax.plot_surface(X_3d, T_3d, T_pred_3d, cmap='coolwarm', alpha=0.8)

    ax.set_xlabel('Position x (m)', fontsize=14)
    ax.set_ylabel('Time t (s)', fontsize=14)
    ax.set_zlabel('Temperature T (K)', fontsize=14)
    ax.set_title('3D Temperature Distribution\n(PINN Solution)',
                 fontsize=16, fontweight='bold')

    cbar = fig.colorbar(surf, shrink=0.5, aspect=10)
    cbar.set_label('Temperature (K)', fontsize=14)

    plt.tight_layout()
    plt.savefig('cooling_fin_3d_surface.png', dpi=300, bbox_inches='tight')
    plt.show()



# =============================================================================
# HEAT TRANSFER ANALYSIS
# =============================================================================
def analyze_heat_transfer_characteristics(model):
    """
    Compute and report key dimensionless numbers and engineering performance
    metrics derived from the trained PINN predictions.

    Metrics include:
        - Convective and diffusive time constants
        - Biot number (Bi): ratio of convective to conductive resistance
        - Fourier number (Fo): dimensionless time for diffusion
        - Final temperature distribution and average temperature
        - Fin effectiveness and cooling efficiency

    Args:
        model : Trained CoolingFinPINN instance
    """
    print("\n" + "="*70)
    print("HEAT TRANSFER ANALYSIS RESULTS")
    print("="*70)

    # Time constants characterizing dominant physical processes
    tau_conv = rho * c_p / (h * P)   # Convective time constant [s]
    tau_diff = L**2 / alpha           # Diffusive time constant [s]

    print(f"Convective time constant τ_conv = {tau_conv:.1f} s")
    print(f"Diffusive time constant τ_diff  = {tau_diff:.1f} s")

    # Biot number: compares internal conduction resistance to surface convection
    # Bi << 0.1 → lumped model valid; Bi > 0.1 → distributed model required
    Bi = h * L / k
    print(f"Biot number Bi = {Bi:.4f}")
    if Bi < 0.1:
        print("  → Lumped capacitance model would be adequate")
    else:
        print("  → Distributed model necessary (temperature gradients significant)")

    # Fourier number: dimensionless measure of elapsed diffusion time
    Fo = alpha * t_max / L**2
    print(f"Fourier number at t_max: Fo = {Fo:.2f}")

    # Predict steady-state (final) temperature distribution at t = t_max
    x_final = np.linspace(0, L, 101).reshape(-1, 1)
    t_final = t_max * np.ones_like(x_final)

    T_final = model.predict(
        tf.constant(x_final, dtype=tf.float32),
        tf.constant(t_final, dtype=tf.float32)
    ).numpy()

    print(f"\nTemperature Analysis at t = {t_max} s:")
    print(f"  Base (x=0): {T_final[0,0]:.1f} K  ({T_final[0,0]-273.15:.1f}°C)")
    print(f"  Tip  (x=L): {T_final[-1,0]:.1f} K ({T_final[-1,0]-273.15:.1f}°C)")
    print(f"  Base-to-tip drop: {T_final[0,0] - T_final[-1,0]:.1f} K")

    # Average fin temperature at final time
    T_avg = np.mean(T_final)
    print(f"  Average temperature: {T_avg:.1f} K ({T_avg-273.15:.1f}°C)")

    # Fin effectiveness: fraction of maximum possible temperature above ambient
    effectiveness = (T_avg - T_inf) / (T_base - T_inf)
    print(f"  Fin effectiveness: {effectiveness:.3f}")

    print(f"\nFin Performance Summary:")
    print(f"  Initial temperature : {T_initial:.1f} K ({T_initial-273.15:.1f}°C)")
    print(f"  Ambient temperature : {T_inf:.1f} K ({T_inf-273.15:.1f}°C)")
    print(f"  Max temperature diff: {T_initial - T_inf:.1f} K")
    print(f"  Final avg cooling   : {T_initial - T_avg:.1f} K")
    print(f"  Cooling efficiency  : {(T_initial - T_avg)/(T_initial - T_inf)*100:.1f}%")


# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    """
    Orchestrates the full PINN workflow:
        1. Define network architecture and domain bounds
        2. Generate training data (PDE, IC, BC points)
        3. Initialize and train the PINN model
        4. Generate visualizations of the learned solution
        5. Report heat transfer performance metrics
    """
    print("Physics-Informed Neural Network for Cooling Fin Heat Transfer")
    print("="*70)

    # Network architecture: 2 inputs (x, t), 4 hidden layers, 1 output (T)
    layers = [2, 64, 64, 64, 64, 1]

    # Physical domain bounds: x ∈ [0, L], t ∈ [0, t_max]
    lb = [0.0, 0.0]
    ub = [L, t_max]

    # Generate all training data
    training_data = generate_training_data()

    # Initialize PINN with defined architecture and domain
    print(f"\nInitializing PINN with architecture: {layers}")
    model = CoolingFinPINN(layers, lb, ub)

    # Train the model for 20,000 epochs with initial lr = 0.001
    loss_history = model.train(
        training_data['x_pde'], training_data['t_pde'],
        training_data['x_ic'],  training_data['t_ic'],  training_data['T_ic'],
        training_data['x_bc1'], training_data['t_bc1'], training_data['T_bc1'],
        training_data['x_bc2'], training_data['t_bc2'],
        epochs=20000, lr=0.001
    )

    # Generate all visualization plots
    print("\nGenerating comprehensive visualizations...")
    create_visualizations(model)

    # Compute and display engineering performance metrics
    analyze_heat_transfer_characteristics(model)

    print(f"\nFinal training loss: {loss_history[-1]:.2e}")
    print("\nVisualization files saved:")
    print("  - cooling_fin_temperature_evolution.png")
    print("  - cooling_fin_heatmap.png")
    print("  - cooling_fin_training_convergence.png")
    print("  - cooling_fin_temporal_evolution.png")
    print("  - cooling_fin_3d_surface.png")

    return model

# Entry point: only runs when script is executed directly (not when imported)
if __name__ == "__main__":
    model = main()


Thermal diffusivity α = 8.23e-05 m²/s
Convective parameter β = 2.06e-06 1/s
Physics-Informed Neural Network for Cooling Fin Heat Transfer
Generated training data:
  PDE points: 10000
  IC points:  1000
  BC points:  1000

Initializing PINN with architecture: [2, 64, 64, 64, 64, 1]
Starting PINN training...
Epoch     0: Loss = 1.19e+07, PDE = 3.69e+07, IC = 1.49e+03, BC = 1.19e+07, Time = 2.0s
Epoch  2000: Loss = 1.44e+03, PDE = 1.48e+09, IC = 6.42e+02, BC = 7.84e+02, Time = 516.5s
Epoch  4000: Loss = 4.38e+02, PDE = 1.51e+09, IC = 1.94e+02, BC = 2.29e+02, Time = 1012.5s
Epoch  6000: Loss = 2.52e+02, PDE = 9.18e+08, IC = 1.20e+02, BC = 1.22e+02, Time = 1534.8s
Epoch  8000: Loss = 1.42e+02, PDE = 5.67e+08, IC = 7.91e+01, BC = 5.69e+01, Time = 2115.3s
Epoch 10000: Loss = 1.01e+02, PDE = 3.74e+08, IC = 6.18e+01, BC = 3.53e+01, Time = 2595.1s
Epoch 12000: Loss = 6.70e+01, PDE = 2.69e+08, IC = 4.40e+01, BC = 2.03e+01, Time = 3077.7s
Epoch 14000: Loss = 5.09e+01, PDE = 2.22e+08, IC = 3.49e+01

In [25]:
from google.colab import files
import os

files_to_download = [
    'cooling_fin_temperature_evolution.png',
    'cooling_fin_heatmap.png',
    'cooling_fin_training_convergence.png',
    'cooling_fin_temporal_evolution.png',
    'cooling_fin_3d_surface.png'
]

for filename in files_to_download:
    if os.path.exists(filename):
        files.download(filename)
    else:
        print(f"Warning: File not found for download: {filename}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>